# Pitchside ball finder

Finds **goal candidates** in a match video by tracking the ball, then gives you a list of times to paste into Pitchside's *Enter goal times* box.

**You only do three things:** switch on a GPU, run the cells top to bottom, and download the zip at the end.

1. **Runtime → Change runtime type → T4 GPU** (free). Without a GPU this is far too slow.
2. Run each cell in order (press the play button, or Shift+Enter).
3. Cell 3 asks for your match video. For a big file, put it in Google Drive first and choose *google drive*.

It reports a goal when the ball reaches a goal area **and then sits still on the centre spot** shortly after (the kick-off). Shots and saves that did not lead to a kick-off are listed as *maybe*.

In [ ]:
#@title 1. Install (run once)
!pip -q install ultralytics huggingface_hub
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - go to Runtime > Change runtime type and pick T4 GPU")

In [ ]:
#@title 2. Load the ball finder (just run it)
# ---- Pitchside ball finder: core logic (no model needed to run these functions) ----
import csv, os
import numpy as np
import cv2


def scan_video(path, detect_batch, sample_fps=5.0, batch=16, progress=None, max_seconds=None):
    """Read every Nth frame and run detect_batch(list_of_frames) -> list of (x, y, conf) or None.
    x, y are 0-1 fractions of the frame. Returns (rows, info) where rows = [(t, x, y, conf), ...]."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError("Could not open the video: " + path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    dur = total / fps if total else 0
    step = max(1, round(fps / sample_fps))
    rows, frames, times, i = [], [], [], 0

    def flush():
        if not frames:
            return
        for t, d in zip(times, detect_batch(frames)):
            if d is not None:
                rows.append((t, float(d[0]), float(d[1]), float(d[2])))
        frames.clear(); times.clear()

    while True:
        if not cap.grab():
            break
        if i % step == 0:
            ok, fr = cap.retrieve()
            if ok:
                frames.append(fr); times.append(i / fps)
                if len(frames) >= batch:
                    flush()
                    if progress:
                        progress(i / fps, dur, len(rows))
        i += 1
        if max_seconds and i / fps > max_seconds:
            break
    flush()
    cap.release()
    return rows, {"fps": fps, "duration": dur if dur else i / fps, "step": step}


def clean_track(rows, dur):
    """Remove detections that sit on one spot for most of the match (corner flags, a spare ball,
    line markings) and one-off jumps. Returns an array of [t, x, y, conf]."""
    pts = np.array(sorted(rows), dtype=float).reshape(-1, 4)
    if len(pts) < 20:
        return pts
    cx = np.floor(pts[:, 1] / 0.02).astype(int); cy = np.floor(pts[:, 2] / 0.02).astype(int)
    keep = np.ones(len(pts), bool)
    for gx in np.unique(cx):
        for gy in np.unique(cy[cx == gx]):
            m = (abs(cx - gx) <= 1) & (abs(cy - gy) <= 1)
            if m.sum() >= 30:  # a fixed false spot shows up all through the match; a centre spot only at kick-offs
                present = len(set(np.minimum(9, (pts[m, 0] / max(dur, 1) * 10).astype(int))))
                if present >= 7:
                    keep &= ~((cx == gx) & (cy == gy))
    pts = pts[keep]
    out = []
    for i, p in enumerate(pts):
        if 0 < i < len(pts) - 1:
            a, b = pts[i - 1], pts[i + 1]
            d1 = np.hypot(*(p[1:3] - a[1:3])); d2 = np.hypot(*(p[1:3] - b[1:3])); d3 = np.hypot(*(a[1:3] - b[1:3]))
            if p[0] - a[0] < 1.5 and b[0] - p[0] < 1.5 and d1 > 0.25 and d2 > 0.25 and d3 < 0.15:
                continue
        out.append(p)
    return np.array(out).reshape(-1, 4)


def suggest_zones(pts, end_frac=0.07):
    """Guess the two goal areas from where the ball travels: the outer slice of its left-right range,
    around the middle of the pitch height. Returns (left, right, centre) as (x0, x1, y0, y1)."""
    x, y = pts[:, 1], pts[:, 2]
    lo, hi = np.percentile(x, [0.5, 99.5]); span = hi - lo
    yc = float(np.median(y)); yr = np.percentile(y, 95) - np.percentile(y, 5)
    y0, y1 = yc - 0.2 * yr, yc + 0.2 * yr
    left = (0.0, lo + end_frac * span, y0, y1)      # to the frame edge: anything past the end line is in the net
    right = (hi - end_frac * span, 1.0, y0, y1)
    mid = (lo + hi) / 2
    centre = (mid - 0.08 * span, mid + 0.08 * span, yc - 0.22 * yr, yc + 0.22 * yr)
    return left, right, centre


def _inside(pts, z):
    return (pts[:, 1] >= z[0]) & (pts[:, 1] <= z[1]) & (pts[:, 2] >= z[2]) & (pts[:, 2] <= z[3])


def _restarts(pts, centre, still=0.02):
    """Times when the ball sits still in the centre circle (>= 3 sightings over >= 1.5 s)."""
    idx = np.where(_inside(pts, centre))[0]
    out = []
    for a in range(len(idx)):
        t = pts[idx[a], 0]
        if out and t - out[-1] < 10:
            continue
        grp = [j for j in idx[a:] if pts[j, 0] - t <= 4.0]
        if len(grp) >= 3 and pts[grp[-1], 0] - t >= 1.5 and np.ptp(pts[grp, 1]) < still and np.ptp(pts[grp, 2]) < still * 1.5:
            out.append(float(t))
    return out


def find_candidates(pts, zones, min_hits=2, restart_window=(8, 120)):
    """A goal candidate = ball seen in a goal area, then sitting still on the centre spot soon after.
    Each centre restart is given to the most recent goal-area visit before it, so a save followed
    by a goal does not count twice. 'restart' False means the ball reached a goal area only
    (shot, save, goal kick) and is reported as 'maybe'."""
    left, right, centre = zones
    clusters = []
    for name, z in (("left", left), ("right", right)):
        idx = np.where(_inside(pts, z))[0]
        start = 0
        for k in range(1, len(idx) + 1):
            if k == len(idx) or pts[idx[k], 0] - pts[idx[k - 1], 0] > 4.0:
                grp = idx[start:k]; start = k
                if len(grp) >= min_hits:
                    clusters.append({"time": float(pts[grp[0], 0]), "t1": float(pts[grp[-1], 0]), "end": name,
                                     "hits": int(len(grp)), "restart_at": None})
    for r in _restarts(pts, centre):
        ok = [c for c in clusters if c["t1"] + restart_window[0] <= r <= c["t1"] + restart_window[1]]
        if ok:
            best = max(ok, key=lambda c: c["t1"])
            if best["restart_at"] is None:
                best["restart_at"] = r
    for c in clusters:
        c["restart"] = c["restart_at"] is not None
        c["score"] = round(0.4 + 0.2 * min(c["hits"], 6) / 6 + (0.4 if c["restart"] else 0), 2)
    clusters.sort(key=lambda c: c["time"])
    merged = []
    for c in clusters:  # near-duplicates (e.g. save then goal-kick), keep the one with a restart
        if merged and c["time"] - merged[-1]["time"] < 25:
            if c["restart"] and not merged[-1]["restart"]:
                merged[-1] = c
            continue
        merged.append(c)
    return merged


def mmss(t):
    t = int(round(t)); h, m, s = t // 3600, (t % 3600) // 60, t % 60
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


def write_outputs(rows, pts, cands, outdir):
    os.makedirs(outdir, exist_ok=True)
    with open(os.path.join(outdir, "ball_track.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["seconds", "x", "y", "confidence"])
        for r in pts: w.writerow([f"{r[0]:.2f}", f"{r[1]:.4f}", f"{r[2]:.4f}", f"{r[3]:.3f}"])
    with open(os.path.join(outdir, "goal_candidates.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["time", "seconds", "goal_end", "ball_in_goal_area_hits", "centre_restart_at", "likely_goal", "score"])
        for c in cands:
            w.writerow([mmss(c["time"]), f"{c['time']:.1f}", c["end"], c["hits"], mmss(c["restart_at"]) if c["restart"] else "", "yes" if c["restart"] else "maybe", c["score"]])
    with open(os.path.join(outdir, "goal_times.txt"), "w") as f:
        f.write("\n".join(mmss(c["time"]) for c in cands if c["restart"]) + "\n")


# ---- model glue (needs ultralytics; runs on the Colab GPU) ----
def load_ball_model(repo_ids=("acatorcini/yolov9-soccer-ball", "Ssssad542/soccana"), local_path=None):
    """Load a football ball detector. Uses local_path if given, otherwise downloads from Hugging Face
    (first repo that works). Returns (model, description)."""
    from ultralytics import YOLO
    if local_path:
        return YOLO(local_path), "local file " + local_path
    from huggingface_hub import hf_hub_download, list_repo_files
    errors = []
    for repo in repo_ids:
        try:
            files = list_repo_files(repo)
            weights = [f for f in files if f.lower().endswith(".pt")] or [f for f in files if f.lower().endswith(".onnx")]
            if not weights:
                errors.append(f"{repo}: no .pt or .onnx file"); continue
            weights.sort(key=lambda f: ("best" not in f.lower(), len(f)))
            p = hf_hub_download(repo, weights[0])
            return YOLO(p), f"{repo} / {weights[0]}"
        except Exception as e:  # try the next one
            errors.append(f"{repo}: {e}")
    raise RuntimeError("Could not load a ball model:\n" + "\n".join(errors))


def make_yolo_detector(model, conf=0.15, imgsz=1280, only_ball=True):
    """Wrap an Ultralytics model as detect_batch(frames) -> [(x, y, conf) or None, ...]."""
    names = getattr(model, "names", {}) or {}
    ids = [i for i, n in names.items() if str(n).lower().endswith("ball")]
    classes = ids if (only_ball and ids) else None

    def detect_batch(frames):
        res = model.predict(frames, imgsz=imgsz, conf=conf, classes=classes, verbose=False)
        out = []
        for r in res:
            b = r.boxes
            if b is None or len(b) == 0:
                out.append(None); continue
            k = int(b.conf.argmax())
            x, y, _, _ = b.xywhn[k].tolist()
            out.append((x, y, float(b.conf[k])))
        return out
    return detect_batch

In [ ]:
#@title 3. Get your match video
SOURCE = "upload"  #@param ["upload", "google drive"]
DRIVE_PATH = "/content/drive/MyDrive/match.mp4"  #@param {type:"string"}
import os
if SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    VIDEO = "/content/" + list(up.keys())[0]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = DRIVE_PATH
cap = cv2.VideoCapture(VIDEO)
print("Video:", VIDEO, "| %.1f minutes | %dx%d | %.1f fps" % (cap.get(cv2.CAP_PROP_FRAME_COUNT) / (cap.get(cv2.CAP_PROP_FPS) or 25) / 60,
      cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT), cap.get(cv2.CAP_PROP_FPS)))
cap.release()

In [ ]:
#@title 4. Load the model and test the first 3 minutes
SAMPLE_FPS = 5      #@param {type:"number"}
CONFIDENCE = 0.15   #@param {type:"number"}
IMAGE_SIZE = 1280   #@param {type:"integer"}
MODEL_FILE = ""     #@param {type:"string"}
# MODEL_FILE: leave empty to download a football ball model from Hugging Face. To use your own, upload a .pt file
# to Colab (folder icon on the left) and type its path here, e.g. /content/ball.pt

import time
model, desc = load_ball_model(local_path=MODEL_FILE or None)
print("Model:", desc, "| classes:", model.names)
detect = make_yolo_detector(model, conf=CONFIDENCE, imgsz=IMAGE_SIZE)

# quick test on the first 3 minutes so you know it can see the ball before the long run
t0 = time.time()
test_rows, info = scan_video(VIDEO, detect, sample_fps=SAMPLE_FPS, max_seconds=180)
n_samples = int(180 * SAMPLE_FPS)
print("Quick test: ball seen in %d of ~%d samples (%.0f%%) in %.0fs" % (len(test_rows), n_samples, 100 * len(test_rows) / n_samples, time.time() - t0))
print("Estimated time for the whole match: about %.0f minutes" % ((time.time() - t0) / 180 * info["duration"] / 60))
print("Rough guide: 20%+ is workable, under 5% means this model cannot see the ball well on your footage.")

In [ ]:
#@title 5. Scan the whole match (takes a while)
from tqdm.auto import tqdm
bar = tqdm(total=100, desc="Scanning match")
last = [0]
def prog(t, dur, n):
    pct = int(100 * t / max(dur, 1)); bar.update(pct - last[0]); last[0] = pct; bar.set_postfix(ball_sightings=n)
rows, info = scan_video(VIDEO, detect, sample_fps=SAMPLE_FPS, progress=prog)
bar.close()
print("Done:", len(rows), "ball sightings in", round(info["duration"] / 60, 1), "minutes of video")

In [ ]:
#@title 6. Find goal candidates
import matplotlib.pyplot as plt
ZONES = None   #@param {type:"raw"}
# Leave ZONES as None to let it guess the goal areas and centre spot from where the ball travels.
# To set them yourself: ZONES = ((x0,x1,y0,y1), (x0,x1,y0,y1), (x0,x1,y0,y1)) for left goal, right goal, centre,
# all as fractions of the picture (0-1; 0,0 is the top left).

pts = clean_track(rows, info["duration"])
zones = ZONES or suggest_zones(pts)
cands = find_candidates(pts, zones)

cap = cv2.VideoCapture(VIDEO); cap.set(cv2.CAP_PROP_POS_MSEC, 60000); ok, frame = cap.read(); cap.release()
fig, ax = plt.subplots(figsize=(11, 6))
if ok: ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), extent=(0, 1, 1, 0), aspect="auto")
ax.scatter(pts[:, 1], pts[:, 2], s=3, c="yellow", alpha=0.35)
for z, col, lab in zip(zones, ("cyan", "cyan", "magenta"), ("left goal area", "right goal area", "centre spot")):
    ax.add_patch(plt.Rectangle((z[0], z[2]), z[1] - z[0], z[3] - z[2], fill=False, ec=col, lw=2)); ax.text(z[0], z[2], lab, color=col, fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_title("Yellow dots = where the ball was seen. Boxes = goal areas and centre spot")
plt.show()
print("If the boxes are not on the goals and centre spot, set ZONES above and run this cell again.")
print()
print("%-8s %-6s %-5s %-14s %s" % ("time", "end", "hits", "kick-off at", "verdict"))
for c in cands:
    print("%-8s %-6s %-5d %-14s %s" % (mmss(c["time"]), c["end"], c["hits"], mmss(c["restart_at"]) if c["restart"] else "-", "LIKELY GOAL" if c["restart"] else "maybe (shot/save?)"))
print()
print("Paste into Pitchside > Enter goal times:")
print(", ".join(mmss(c["time"]) for c in cands if c["restart"]) or "(none found)")

In [ ]:
#@title 7. Save and download the results
import zipfile, shutil
OUT = "/content/pitchside_ball"
write_outputs(rows, pts, cands, OUT)
fig, axes = plt.subplots(2, 4, figsize=(16, 6)); axes = axes.ravel()
cap = cv2.VideoCapture(VIDEO)
for a in axes: a.axis("off")
for a, c in zip(axes, [c for c in cands if c["restart"]] + [c for c in cands if not c["restart"]]):
    cap.set(cv2.CAP_PROP_POS_MSEC, (c["time"] + 1) * 1000); ok, fr = cap.read()
    if ok: a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    a.set_title(mmss(c["time"]) + (" LIKELY" if c["restart"] else " maybe"), fontsize=10)
cap.release(); plt.tight_layout(); plt.show()
shutil.make_archive(OUT, "zip", OUT)
from google.colab import files
files.download(OUT + ".zip")
print("Downloaded pitchside_ball.zip: goal_times.txt (paste into Pitchside), goal_candidates.csv, ball_track.csv")